[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/math/spherical_synchronization/spherical_synchronization.ipynb)

# Spherical Synchronization

This example simulates spherical synchronization, the Kuramoto model on a sphere in which a population of turning directions attracts itself toward a common direction. When every oscillator shares the same natural rotation, Lipton, Mirollo, and Strogatz (2021) showed that the entire population evolves through a single Möbius transformation of the sphere, reducing an arbitrary number of oscillators to one six-parameter state.

In spacetime conformal geometry, each point of the sphere is modelled as a null ray `direction + time` and each circle as a planar cut whose points satisfy `cut & point == 0`. One rotor carries points and circles together via `rotor >> seeds` and `rotor >> grid`, preserving every circle and intersection angle as the population gathers. Its instantaneous bivector rate combines the shared rotation with a boost `mean.commutator(time)` toward the mean direction, which acts tangentially once null rays are rescaled back to unit time.

In Gibbs–Heaviside vector notation on $\mathbb{R}^3$, each unit vector evolves as $\dot{\mathbf{x}}_i = \boldsymbol{\omega} \times \mathbf{x}_i + K\,(\bar{\mathbf{x}} - (\bar{\mathbf{x}} \cdot \mathbf{x}_i)\,\mathbf{x}_i)$ with $\bar{\mathbf{x}} = \frac{1}{N}\sum_{j=1}^N \mathbf{x}_j$, or under the Watanabe–Strogatz ansatz via a boost vector $\mathbf{a}(t) \in B^3$ and rotation matrix $\mathbf{R}(t) \in \mathrm{SO}(3)$ coupled through $\mathbf{x}_i(t) = \frac{(1 - \|\mathbf{a}\|^2)\,\mathbf{R}\mathbf{x}_i(0) + (1 + 2\,\mathbf{a}\cdot\mathbf{R}\mathbf{x}_i(0) + \|\mathbf{a}\|^2)\,\mathbf{a}}{1 + 2\,\mathbf{a}\cdot\mathbf{R}\mathbf{x}_i(0) + \|\mathbf{a}\|^2\,\|\mathbf{R}\mathbf{x}_i(0)\|^2}$.

In [ ]:
# In Colab, install numga and its examples from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from collections.abc import Iterator
from itertools import islice

import numpy as np

from numga import NumpyContext, concatenate, stack
from numga.algebras import STA as ga
from examples.util.animation import save_figure, show_states
from examples.math.spherical_synchronization import render

# Spacetime algebra x+y+z+t- carries the conformal geometry of the unit sphere.
mv = NumpyContext(ga).multivector
# Circles are cuts of the sphere, vectors; points are null rays, antivectors.
Cut = ga.gatype.vector()                                              # Vector+
Point = ga.gatype.antivector()                                        # Antivector-
Bivector = ga.gatype.bivector()                                       # Bivector+
Rotor = ga.gatype.rotor()                                             # Rotor+
# The rays' time direction pairs to one with `mv.t`; the dual carries the minus sign of `t`'s square.
time = -mv.t.dual()                                                   # [] Point-

## Initial population and circle grid

A golden-angle spiral places directions nearly uniformly over the sphere before a small initial boost nudges them toward one side, while meridians and latitude parallels mark circles carried alongside the particles.

In [ ]:
n_particles = 192
n_meridians = 4
n_parallels = 5
n_cuts = n_meridians + n_parallels
initial_bias = 0.24
golden_angle = np.pi * (3 - np.sqrt(5))
longitudes = golden_angle * np.arange(n_particles) % (2 * np.pi)
latitudes = np.arcsin(np.linspace(-1 + 1 / n_particles, 1 - 1 / n_particles, n_particles))
meridian_angles = np.linspace(0, np.pi, n_meridians, endpoint=False)
parallel_angles = np.linspace(-np.pi / 3, np.pi / 3, n_parallels)

In [ ]:
def directions(points: Point) -> Point:
    """Rescaling null rays (`[...] Point-`) to unit time and subtracting `time` recovers their unit
    spatial directions on the sphere."""
    return points / (mv.t & points) - time


def sphere_points(longitudes: np.ndarray, latitudes: np.ndarray) -> Point:
    """Turning the reference direction along `longitudes` and `latitudes` places unit spatial directions
    on the sphere (`[...] Point-`)."""
    longitude = (-mv.xy * (longitudes / 2)).exp()                     # [...] Rotor+
    latitude = (-mv.xz * (latitudes / 2)).exp()                       # [...] Rotor+
    return (longitude * latitude) >> mv.x.dual()                      # [...] Point-


seeds = sphere_points(longitudes, latitudes) + time                   # [n_particles] Point-
# Each cut intersects the unit sphere in a circle where its pairing with null rays vanishes.
meridian_turns = (-mv.xy * (meridian_angles / 2)).exp()               # [n_meridians] Rotor+
meridians = meridian_turns >> mv.y                                    # [n_meridians] Cut+
parallels = mv.z + mv.t * np.sin(parallel_angles)                     # [n_parallels] Cut+
grid = concatenate((meridians, parallels), axis=0)                    # [n_cuts] Cut+
initial = (-mv.xt * initial_bias).exp()                               # [] Rotor+
initial_population = directions(initial >> seeds)                     # [n_particles] Point-
initial_grid = initial >> grid                                        # [n_cuts] Cut+

save_figure(render.sphere_figure(initial_population, initial_grid), "spherical_synchronization");

Coloured dots show individual directions on the sphere and teal curves show the meridian and parallel cuts, with the far hemisphere faded behind the near hemisphere.

## Synchronization under one Möbius motion

Each oscillator turns toward the population's mean direction while rotating around the vertical axis. On null rays that tangential pull is the boost `mean.commutator(time)`, which adds directly to the shared rotation in one bivector rate updated as the population moves. Rendering the animation takes about three seconds.

In [ ]:
coupling = 1.2
rotation_speed = 0.25
span = 5.0
n_steps = 150
dt = span / n_steps
duration_ms = 40
rotation = -mv.xy * rotation_speed                                    # [] Bivector+

In [ ]:
def flow_rate(rotor: Rotor, seeds: Point, rotation: Bivector, coupling: float) -> Bivector:
    """The instantaneous bivector rate combines the shared `rotation` with a boost toward the
    population's mean direction (`[] Bivector+`)."""
    population = directions(rotor >> seeds)                           # [...] Point-
    mean = population.mean(axis=-1)                                   # [] Point-
    # Rescaling each null ray back to the sphere turns this boost into tangential attraction.
    return rotation - coupling * mean.commutator(time)                # [] Bivector+


def motion(initial: Rotor, seeds: Point, rotation: Bivector, coupling: float, dt: float) -> Iterator[Rotor]:
    """Yields the common Möbius rotor (`[] Rotor+`) from `initial` onward, advancing each step by the
    midpoint `flow_rate`."""
    rotor = initial                                                   # [] Rotor+
    while True:
        yield rotor
        halfway = (flow_rate(rotor, seeds, rotation, coupling) * (dt / 4)).exp() * rotor   # [] Rotor+
        rotor = (flow_rate(halfway, seeds, rotation, coupling) * (dt / 2)).exp() * rotor   # [] Rotor+


# One rotor carries the population and the circle grid together at every step.
rotors = islice(motion(initial, seeds, rotation, coupling, dt), n_steps + 1)
states = ((directions(rotor >> seeds), rotor >> grid) for rotor in rotors)
recorded_states = show_states(
    states, render.frames, name="spherical_synchronization", duration_ms=duration_ms, n_frames=n_steps + 1,
)

As the coloured dots gather toward a single rotating cluster, the teal meridians and parallels remain exact circles meeting at right angles because one invertible Möbius rotor deforms both the oscillators and the grid.

In [ ]:
# checks
# Initial rays lie on the null cone, every streamed frame stays on the sphere, and the spread collapses.
assert ((initial_population + time).squared().abs() < 1e-10).all()
population = stack([places for places, _ in recorded_states])         # [n_steps + 1, n_particles] Point-
assert ((population + time).squared().abs() < 1e-10).all()
spread = (population - population.mean(axis=-1)[:, None]).squared().mean(axis=-1).abs()   # [n_steps + 1] Scalar+
assert spread[-1] < 0.05 * spread[0]